# Imbalanced TelecomTS Split (Scenario 2)

> **This notebook is the imbalanced-split twin of `../TelecomTS/Foundation_Models_TelecomTS.ipynb`.** The data layout under `data/` and `data/features_cache/` is identical in shape to the balanced version, but the underlying files are symlinks to the **1000-window TelecomTS split** (`../../split/TelecomTS_*.npz`) and the precomputed Chronos-2 residuals at the project root (`../../X_*_resid_full_norm.npy`).

| Split | Total | Normal | Anomaly | Positive rate |
|-------|------:|-------:|--------:|--------------:|
| Train |   640 |   608  |     32  |        ~5%   |
| Val   |   160 |   152  |      8  |        ~5%   |
| Test  |   200 |   190  |     10  |        ~5%   |

All BCE-trained methods auto-compute `pos_weight = N_neg / N_pos` from the training labels, so the only effective change vs the balanced notebook is the data path. Threshold-tunable methods continue to pick the best-F1 cutoff on the (also-imbalanced) validation split.

Results are written to `results/` in this folder.


# Foundation-Model Baselines on TelecomTS

**Goal.** Add three off-the-shelf time-series foundation models — **MOMENT**, **TOTO**, and **Mantis** — to the TelecomTS anomaly-detection benchmark, evaluated on the **imbalanced 200-window TelecomTS test split** (190 normal / 10 anomaly) — the same split used by every other baseline in this folder.

**Protocol** (matches `TelecomTS_RCA_fair_comparison.ipynb`, which itself follows TelecomTS Appendix F):

* Frozen TSFM backbone + a single `nn.Linear` head trained for binary anomaly detection.
* Adam, `lr=1e-4`, `batch=64`, 10 epochs, BCE-with-logits.
* Train on `train + val` concatenated; evaluate on `test`.
* No fine-tuning of TSFM weights, no class-weighting, no hyperparameter sweep.
* Seed = 42.

**Why this is "weak-but-fair" by design.** The point is to benchmark *what an off-the-shelf TSFM gives you with a vanilla classification head*, not to find the strongest possible TSFM-based detector. The proposed KAC method consumes Chronos-2 residual + uncertainty features *and* a text branch with KPI-level contrastive alignment — none of which these baselines use.

Input shape: `(N, 16, 128)` (16 KPIs × 128 timesteps, 190/10 normal/anom in test).


## 1. Imports & device

In [ ]:
import os
import sys
import types
import importlib.util
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Device: {DEVICE} | torch {torch.__version__}")

REPO_ROOT = Path("..") / ".."
MODELS_DIR = REPO_ROOT / "models"
OUT = Path("results")
OUT.mkdir(parents=True, exist_ok=True)


## 2. Install foundation-model packages

In [ ]:
# Install foundation-model packages (idempotent; --no-deps keeps the venv quiet).
!pip install -q --no-deps momentfm
!pip install -q --no-deps toto-ts
!pip install -q rotary-embedding-torch
!pip install -q --no-deps 'gluonts==0.16.2'
!pip install -q --no-deps mantis-tsfm
print("Packages ready.")


## 3. SSL fix for HuggingFace (macOS)

In [ ]:
# HuggingFace SSL fix for macOS (matches TelecomTS_RCA_fair_comparison.ipynb).
import ssl, certifi, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "-q", "certifi"],
               capture_output=True)
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
ssl._create_default_https_context = ssl._create_unverified_context
print(f"SSL bundle: {certifi.where()}")


## 4. Load data
Same `*_test.npz` used by every other baseline in `results/benchmark_comparison.csv`.

In [ ]:
DATA_DIR = Path("data")
train_d = np.load(DATA_DIR / "TelecomTS_train.npz", allow_pickle=True)
val_d   = np.load(DATA_DIR / "TelecomTS_val.npz",   allow_pickle=True)
test_d  = np.load(DATA_DIR / "TelecomTS_test.npz",  allow_pickle=True)

# X is (N, T=128, C=16); models expect (N, C, T).
X_train = np.concatenate([train_d["X"], val_d["X"]], axis=0).transpose(0, 2, 1).astype(np.float32)
y_train = np.concatenate([train_d["y"], val_d["y"]], axis=0).astype(np.int64)
X_test  = test_d["X"].transpose(0, 2, 1).astype(np.float32)
y_test  = test_d["y"].astype(np.int64)

N_CHANNELS = X_train.shape[1]   # 16
SEQ_LEN    = X_train.shape[2]   # 128
print(f"Train+val: {X_train.shape} (anom ratio {y_train.mean():.3f})")
print(f"Test:      {X_test.shape}  (anom ratio {y_test.mean():.3f})")

X_tr_t = torch.from_numpy(X_train)
y_tr_t = torch.from_numpy(y_train).long()
X_te_t = torch.from_numpy(X_test)
y_te_arr = y_test
train_dl = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)


## 5. Train / eval helpers

In [ ]:
def _auto_pos_weight(train_dl):
    """Compute BCE pos_weight = N_neg / N_pos from a dataloader (~1 on balanced
    datasets, ~14 on Production-style imbalance). Prevents the head from
    collapsing to the majority class."""
    ys = torch.cat([yb for _, yb in train_dl]).float()
    pos = ys.sum().clamp(min=1.0)
    neg = ys.numel() - ys.sum()
    return (neg / pos).to(DEVICE)


def train_binary_head(model, train_dl, n_epochs=10, lr=1e-4, pos_weight=None):
    """Train any nn.Module that returns logits of shape (B, 1) or (B,) under
    BCE-with-logits. If `pos_weight` is None it is auto-computed from the
    dataloader so the same helper works on balanced and imbalanced data."""
    model.to(DEVICE)
    if pos_weight is None:
        pos_weight = _auto_pos_weight(train_dl)
    print(f"  BCE pos_weight = {float(pos_weight):.3f}")
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    for epoch in range(n_epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_dl:
            xb = xb.to(DEVICE)
            yb = yb.float().to(DEVICE)
            opt.zero_grad()
            logits = model(xb).squeeze(-1)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()
            total_loss += float(loss.item()) * len(yb)
            correct += int(((torch.sigmoid(logits) >= 0.5) == (yb >= 0.5)).sum().item())
            total += len(yb)
        if epoch == 0 or (epoch + 1) % 5 == 0:
            print(f"  epoch {epoch+1:2d}/{n_epochs}  loss={total_loss/total:.4f}  acc={correct/total:.3f}")
    return model


def eval_binary_head(model, X_test_t, y_test):
    model.eval()
    scores = []
    with torch.no_grad():
        for j in range(0, len(X_test_t), 64):
            xb = X_test_t[j:j+64].to(DEVICE)
            logits = model(xb).squeeze(-1).detach().cpu().float().numpy()
            scores.append(logits)
    scores = np.concatenate(scores)
    probs = 1.0 / (1.0 + np.exp(-scores))
    preds = (probs >= 0.5).astype(int)
    metrics = {
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall":    recall_score(y_test, preds, zero_division=0),
        "F1":        f1_score(y_test, preds, zero_division=0),
        "Accuracy":  accuracy_score(y_test, preds),
        "AUROC":     roc_auc_score(y_test, probs) if len(np.unique(y_test)) > 1 else float("nan"),
        "AP":        average_precision_score(y_test, probs) if len(np.unique(y_test)) > 1 else float("nan"),
    }
    print("  " + "  ".join(f"{k}={v:.3f}" for k, v in metrics.items()))
    return metrics


# Container that survives partial failures (e.g. one TSFM fails to load).
results = {}
print("Helpers ready.")


## 6. MOMENT (frozen backbone + linear head)

In [ ]:
# ---------------------------------------------------------------- MOMENT ----
# Frozen MOMENT-1-large backbone + classification head. We feed the native
# window length directly (no padding, no input_mask, no seq_len kwarg) -- this
# matches `TelecomTS_RCA_fair_comparison.ipynb` and avoids the ~256x slowdown
# from padding to MOMENT's max seq_len of 512.
try:
    from momentfm import MOMENTPipeline

    moment_path = MODELS_DIR / "MOMENT-1-large"
    moment_src = str(moment_path) if moment_path.is_dir() else "AutonLab/MOMENT-1-large"
    print(f"Loading MOMENT from: {moment_src}")
    moment_pipe = MOMENTPipeline.from_pretrained(
        moment_src,
        model_kwargs={
            "task_name": "classification",
            "n_channels": N_CHANNELS,
            "num_class": 2,
        },
    )
    moment_pipe.init()
    for n, p in moment_pipe.named_parameters():
        if "head" not in n.lower() and "classifier" not in n.lower():
            p.requires_grad = False
    n_train = sum(p.numel() for p in moment_pipe.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in moment_pipe.parameters())
    print(f"MOMENT trainable: {n_train:,} / {n_total:,}")

    class MomentBinary(nn.Module):
        def __init__(self, pipe):
            super().__init__()
            self.pipe = pipe
            self.binary_head = nn.Linear(2, 1)
        def forward(self, x):
            out = self.pipe(x_enc=x)
            return self.binary_head(out.logits)

    moment_model = MomentBinary(moment_pipe)
    # MOMENT's 131k-param classification head needs more steps + a slightly
    # higher head learning rate to converge under pos_weight imbalance.
    print("Training MOMENT head (30 epochs, lr=5e-4) ...")
    moment_model = train_binary_head(moment_model, train_dl, n_epochs=30, lr=5e-4)
    print("Evaluating MOMENT ...")
    results["MOMENT (frozen)"] = eval_binary_head(moment_model, X_te_t, y_te_arr)
except Exception as e:
    print(f"MOMENT failed: {e}")
    import traceback; traceback.print_exc()
    results["MOMENT (frozen)"] = {"error": str(e)}


## 7. TOTO (frozen backbone + linear head over mean-pooled embeddings)

In [ ]:
# ------------------------------------------------------------------ TOTO ----
# Frozen Datadog Toto-Open-Base backbone + linear head over mean-pooled patch
# embeddings. The GluonTS shim mirrors `TelecomTS_RCA_fair_comparison.ipynb`
# (Toto only needs AffineTransformed + StudentT from gluonts.torch).
def _minimal_gluonts_torch_for_toto():
    import gluonts
    root = os.path.join(os.path.dirname(gluonts.__file__), "torch")
    aff_path = os.path.join(root, "distributions", "affine_transformed.py")
    spec = importlib.util.spec_from_file_location(
        "gluonts.torch.distributions.affine_transformed", aff_path)
    aff_mod = importlib.util.module_from_spec(spec)
    sys.modules["gluonts.torch.distributions.affine_transformed"] = aff_mod
    spec.loader.exec_module(aff_mod)
    distpkg = types.ModuleType("gluonts.torch.distributions")
    distpkg.AffineTransformed = aff_mod.AffineTransformed
    sys.modules["gluonts.torch.distributions"] = distpkg
    from torch.distributions import StudentT as TorchStudentT
    st_mod = types.ModuleType("gluonts.torch.distributions.studentT")
    class StudentT(TorchStudentT):
        pass
    st_mod.StudentT = StudentT
    sys.modules["gluonts.torch.distributions.studentT"] = st_mod
    gt = types.ModuleType("gluonts.torch")
    gt.__path__ = [root]
    sys.modules["gluonts.torch"] = gt


def _prepare_gluonts_for_toto():
    for k in list(sys.modules):
        if k == "gluonts.torch" or k.startswith("gluonts.torch."):
            del sys.modules[k]
    try:
        from gluonts.torch.distributions import AffineTransformed   # noqa: F401
        from gluonts.torch.distributions.studentT import StudentT   # noqa: F401
    except Exception:
        _minimal_gluonts_torch_for_toto()


try:
    _prepare_gluonts_for_toto()
    from toto.model.toto import Toto

    toto_path = MODELS_DIR / "Toto-Open-Base-1.0"
    toto_src = str(toto_path) if toto_path.is_dir() else "Datadog/Toto-Open-Base-1.0"
    print(f"Loading Toto from: {toto_src}")
    toto = Toto.from_pretrained(toto_src)
    toto_backbone = toto.model.to(DEVICE)
    toto_backbone.eval()

    # Toto-Open-Base-1.0 uses patch_size=64; the scaler asserts that
    # time_steps is a multiple of patch_size. For datasets with T<64 (e.g.
    # Production T=32) we right-pad with zeros and mark the padded steps
    # invalid via input_padding_mask so they don't contribute to scaling.
    TOTO_PATCH = 64
    target_t = ((SEQ_LEN + TOTO_PATCH - 1) // TOTO_PATCH) * TOTO_PATCH

    def get_toto_emb(model, X_t, batch=8):
        embs = []
        with torch.no_grad():
            for j in range(0, len(X_t), batch):
                xb = X_t[j:j+batch].to(DEVICE)
                b, v, t = xb.shape
                if t < target_t:
                    pad = torch.zeros(b, v, target_t - t, dtype=xb.dtype, device=DEVICE)
                    xb = torch.cat([xb, pad], dim=2)
                pad_mask = torch.ones(b, v, target_t, dtype=torch.bool, device=DEVICE)
                if t < target_t:
                    pad_mask[:, :, t:] = False
                id_mask = torch.zeros(b, v, target_t, dtype=torch.float32, device=DEVICE)
                flat, _, _ = model.backbone(xb, pad_mask, id_mask)
                emb = flat.mean(dim=(1, 2))
                embs.append(emb.detach().cpu().float().numpy())
        return np.vstack(embs)

    print("Extracting Toto embeddings (frozen) ...")
    toto_train_emb = get_toto_emb(toto_backbone, X_tr_t)
    toto_test_emb  = get_toto_emb(toto_backbone, X_te_t)
    print(f"Toto emb shapes: train {toto_train_emb.shape}  test {toto_test_emb.shape}")

    # MLP head: gives TOTO comparable head capacity to MOMENT's built-in
    # classification head and Mantis's 16k-d concatenated linear head.
    # TOTO mean-pools to 768 dim, so a 768->256->1 MLP has ~197k params.
    class TotoMLPHead(nn.Module):
        def __init__(self, in_dim, hidden=256, dropout=0.2):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, 1),
            )
        def forward(self, x):
            return self.net(x)

    toto_head = TotoMLPHead(toto_train_emb.shape[1])
    toto_dl = DataLoader(
        TensorDataset(torch.from_numpy(toto_train_emb), y_tr_t),
        batch_size=64, shuffle=True,
    )
    print("Training Toto head (30 epochs, lr=5e-4) ...")
    toto_head = train_binary_head(toto_head, toto_dl, n_epochs=30, lr=5e-4)

    print("Evaluating Toto ...")
    results["TOTO (frozen)"] = eval_binary_head(
        toto_head, torch.from_numpy(toto_test_emb), y_te_arr,
    )
except Exception as e:
    print(f"Toto failed: {e}")
    import traceback; traceback.print_exc()
    results["TOTO (frozen)"] = {"error": str(e)}


## 8. Mantis (per-channel frozen embeddings + linear head)

In [ ]:
# ---------------------------------------------------------------- Mantis ----
# Mantis-8M is univariate at seq_len=512. We embed each KPI channel
# independently, then concatenate the resulting 256-d vectors and train a
# linear head on top. For shorter windows we right-pad with zeros to 512.
try:
    from mantis.architecture import Mantis8M

    MANTIS_SEQ_LEN = 512
    if SEQ_LEN > MANTIS_SEQ_LEN:
        raise ValueError(f"SEQ_LEN={SEQ_LEN} > Mantis seq_len={MANTIS_SEQ_LEN}; truncate first.")

    mantis_path = MODELS_DIR / "Mantis-8M"
    mantis_src = str(mantis_path) if mantis_path.is_dir() else "paris-noah/Mantis-8M"
    print(f"Loading Mantis from: {mantis_src}")

    # Mantis ships internal tensors that don't move with .to(); pin to CPU.
    mantis_net = Mantis8M(device="cpu")
    mantis_net = mantis_net.from_pretrained(mantis_src)
    mantis_net.eval()

    def get_mantis_emb(net, X_t, batch=16):
        n, c, t = X_t.shape
        pad_amt = MANTIS_SEQ_LEN - t
        all_embs = []
        with torch.no_grad():
            for j in range(0, n, batch):
                xb = X_t[j:j+batch]              # (b, c, t)
                b = xb.shape[0]
                xb_uni = xb.reshape(b * c, 1, t)
                if pad_amt > 0:
                    pad = torch.zeros(b * c, 1, pad_amt, dtype=xb_uni.dtype)
                    xb_uni = torch.cat([xb_uni, pad], dim=2)
                emb = net(xb_uni)                 # (b*c, 256)
                emb = emb.reshape(b, -1)          # (b, c*256)
                all_embs.append(emb.detach().cpu().float().numpy())
        return np.vstack(all_embs)

    print("Extracting Mantis embeddings (per-channel, frozen) ...")
    mantis_train_emb = get_mantis_emb(mantis_net, X_tr_t)
    mantis_test_emb  = get_mantis_emb(mantis_net, X_te_t)
    print(f"Mantis emb shapes: train {mantis_train_emb.shape}  test {mantis_test_emb.shape}")

    class LinearBinary(nn.Module):
        def __init__(self, in_dim):
            super().__init__()
            self.fc = nn.Linear(in_dim, 1)
        def forward(self, x):
            return self.fc(x)

    mantis_head = LinearBinary(mantis_train_emb.shape[1])
    mantis_dl = DataLoader(
        TensorDataset(torch.from_numpy(mantis_train_emb), y_tr_t),
        batch_size=64, shuffle=True,
    )
    print("Training Mantis head ...")
    mantis_head = train_binary_head(mantis_head, mantis_dl, n_epochs=10, lr=1e-4)

    print("Evaluating Mantis ...")
    results["Mantis (frozen)"] = eval_binary_head(
        mantis_head, torch.from_numpy(mantis_test_emb), y_te_arr,
    )
except Exception as e:
    print(f"Mantis failed: {e}")
    import traceback; traceback.print_exc()
    results["Mantis (frozen)"] = {"error": str(e)}


## 9. Summary
Saves to `results/foundation_models_comparison.csv` with the same schema as `benchmark_comparison.csv`, so you can `pd.concat` the two and rebuild your paper table in one line.

In [ ]:
rows = []
for name, m in results.items():
    if "error" in m:
        print(f"  {name:20s}  SKIPPED ({m['error']})")
        continue
    print(f"  {name:20s}  " + "  ".join(f"{k}={v:.3f}" for k, v in m.items()))
    rows.append({"Method": name, **m})

df = pd.DataFrame(rows, columns=["Method", "Precision", "Recall", "F1",
                                 "Accuracy", "AUROC", "AP"])
out_path = OUT / "foundation_models_comparison.csv"
df.to_csv(out_path, index=False)
print(f"\nSaved -> {out_path}")
df
